# 02 - Silver Transformation
### Spark SQL for all transformations


In [0]:
# ------------- STEP 1: Remove nulls ------------------

spark.sql("""
          CREATE OR REPLACE TEMP VIEW remove_nulls AS
          SELECT * 
          FROM bronze_transactions
          WHERE amount IS NOT NULL 
          AND Customer_id IS NOT NULL 
          AND transaction_id IS NOT NULL
""")
spark.sql("""
          SELECT count(*) AS Non_Null 
          FROM remove_nulls
""").show()



---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8330375640481031>, line 3
      1 # ------------- STEP 1: Remove nulls ------------------
----> 3 spark.sql("""
      4           CREATE OR REPLACE TEMP VIEW remove_nulls AS
      5           SELECT * 
      6           FROM bronze_transactions
      7           WHERE amount IS NOT NULL 
      8           AND Customer_id IS NOT NULL 
      9           AND transaction_id IS NOT NULL
     10 """)
     11 spark.sql("""
     12           SELECT count(*) AS Non_Null 
     13           FROM remove_nulls
     14 """).show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    898         _views.append(SubqueryAlias(df._plan, name))
    900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.client.execute_c

In [0]:
#------------ STEP 2 Remove Invalid Negatives (keep refund)-----------------

spark.sql("""
          CREATE OR REPLACE TEMP VIEW remove_negatives AS
          SELECT * 
          FROM remove_nulls
          WHERE amount >= 0
          OR transaction_type ="Refund"
""")
spark.sql("""
          SELECT COUNT(*)
          FROM remove_negatives
""").show()


+--------+
|COUNT(*)|
+--------+
|   49900|
+--------+



In [0]:

#------------ STEP 3: De-Duplication using ROW_NUM ------------------

spark.sql("""
          CREATE OR REPLACE TEMP VIEW de_duplicated AS
          SELECT * EXCEPT (row_number)
          FROM(
            SELECT *,
            row_number() OVER(
              PARTITION BY transaction_id
              order by transaction_timestamp 
            ) AS row_number
            FROM remove_negatives
          )
          WHERE row_number=1          
""")

spark.sql("""
          SELECT COUNT(*) as de_deuplicated_count 
          FROM de_duplicated
""").show()

+--------------------+
|de_deuplicated_count|
+--------------------+
|               48395|
+--------------------+



In [0]:
#------------- STEP 4 : Type Casting and Derived Colums ------------------

spark.sql("""
          CREATE OR REPLACE TEMP VIEW enriched_data AS
          SELECT
            transaction_id,
            customer_id,
            ROUND(CAST(amount AS DOUBLE),2) AS amount,
            currency,
            merchant,
            transaction_type,
            city,
            channel,
            CAST(is_fraud AS INT) AS is_fraud,
            to_timestamp(transaction_timestamp,'yyyy-MM-dd HH:mm:ss') AS transaction_timestamp,
            to_date(transaction_timestamp,'yyyy-MM-dd HH:mm:ss') AS transaction_date,
            hour(to_timestamp(transaction_timestamp)) AS transaction_hour,
            ROUND(CAST(fraud_score AS DOUBLE),2) AS risk_score,
            CASE 
                WHEN amount < 500 THEN 'micro'
                WHEN amount < 5000 THEN 'small'
                WHEN amount < 25000 THEN 'medium'
                WHEN amount < 100000 THEN 'large'
                ELSE 'high_value'
            END AS amount_bucket,
            CASE
                WHEN dayofweek(to_date(transaction_timestamp, 'yyyy-MM-dd HH:mm:ss'))
                IN (1,7) THEN 1 
                ELSE 0 
            END AS is_weekend,
            CASE 
                WHEN fraud_score >= 75 THEN 'high_risk'
                WHEN fraud_score >= 40 THEN 'medium_risk'
                ELSE 'low_risk'
            END AS risk_category,
            current_timestamp() AS silver_processed_TS
            FROM de_duplicated
          """)
spark.sql("""
          SELECT count(*) as FINAL_Silver_Count FROM enriched_data
          """).show()

+------------------+
|FINAL_Silver_Count|
+------------------+
|             48395|
+------------------+



In [0]:
#------------STEP 5 :  Preview Enriched Data -----------------
spark.sql("""
          SELECT 
          transaction_id, transaction_type,amount,amount_bucket, risk_category, is_fraud,transaction_date, transaction_hour
          FROM enriched_data LIMIT 10 
          """).show()

+--------------+----------------+--------+-------------+-------------+--------+----------------+----------------+
|transaction_id|transaction_type|  amount|amount_bucket|risk_category|is_fraud|transaction_date|transaction_hour|
+--------------+----------------+--------+-------------+-------------+--------+----------------+----------------+
|   TXN00000001|         Deposit|10733.43|       medium|     low_risk|       0|      2025-11-13|               4|
|   TXN00000002|          Refund|16399.17|       medium|     low_risk|       0|      2025-02-05|              15|
|   TXN00000003|        Withdraw|28706.85|        large|  medium_risk|       0|      2025-06-10|               5|
|   TXN00000004|        Transfer|21624.66|       medium|     low_risk|       0|      2025-10-11|               2|
|   TXN00000005|        Purchase|29551.86|        large|    high_risk|       0|      2025-05-07|              19|
|   TXN00000006|         Deposit| 8599.69|       medium|  medium_risk|       1|      202

In [0]:
# ---- STEP 6: Data quality assertion ----------------
quality_check = spark.sql("""
    SELECT
        SUM(CASE WHEN transaction_id IS NULL THEN 1 ELSE 0 END) AS null_txn_ids,
        SUM(CASE WHEN customer_id IS NULL    THEN 1 ELSE 0 END) AS null_cust_ids,
        SUM(CASE WHEN amount < 0 AND transaction_type != 'refund'
                 THEN 1 ELSE 0 END)                             AS invalid_negatives
    FROM enriched_data
""")
quality_check.show()

+------------+-------------+-----------------+
|null_txn_ids|null_cust_ids|invalid_negatives|
+------------+-------------+-----------------+
|           0|            0|               94|
+------------+-------------+-----------------+



In [0]:
#-------- STEP 7 : Writing/Cteating Silver Table -----------------
spark.table("enriched_data").write.format("delta").mode("overwrite").partitionBy("transaction_date").saveAsTable("silver_transactions")
print(f"Silver Transaction written: {spark.table('silver_transactions').count()} rows")

Silver Transaction written: 48395 rows


In [0]:
#------------ STEP 8 : Clean Silver Customer Table ----------------

spark.sql("""
           CREATE OR REPLACE TEMP VIEW customer_silver_clean AS 
           SELECT 
           customer_id,
           customer_name,
           city,
           account_type,
           round(cast(balance as double),2) AS account_balance,
           status,
           TO_DATE(creation_date,'yyyy-MM-dd') AS account_opened_date,
           current_timestamp() AS silver_processed_ts
           FROM bronze_customers
           WHERE customer_id is NOT NULL
           """)

spark.table("customer_silver_clean").write.format("delta").mode("overwrite").saveAsTable("silver_customers")

print(f"Silver Customers written: {spark.table('silver_customers').count()} rows")

Silver Customers written: 500 rows


In [0]:
#------------------- Bronze Vs Silver Transactions Comparision ---------------------------

spark.sql("""
          SELECT 'bronze_transactions' AS layer , count (*) AS row_count
          FROM bronze_transactions
          UNION ALL
          SELECT 'silver_transactions' , count(*)
          FROM silver_transactions
          """).show()

+-------------------+---------+
|              layer|row_count|
+-------------------+---------+
|bronze_transactions|    51563|
|silver_transactions|    48395|
+-------------------+---------+

